# HARDA — YOLOv8 fine-tuning notebook (Progress 2)

**Goal:** train a 3-class YOLOv8s model for the HARDA hazard taxonomy:
`pothole`, `faded_lane_marking`, `uneven_surface`.

**Why this notebook exists:** the current production weights
(`ml/weights/pothole_yolov8s.pt`) cover only the pothole class. This notebook
produces `harda_v1.pt` — drop-in replacement that handles all three.

**Recommended environment:** Google Colab with a T4 GPU runtime (free tier is
enough). Total run time: ~2–3 hours for 50 epochs on a ~3 000-image dataset.

## Run order
1. Set the Colab runtime to GPU (Runtime → Change runtime type → T4 GPU)
2. Run all cells in order
3. Download `runs/detect/harda_v1/weights/best.pt` at the end
4. Drop it into `ml/weights/harda_v1.pt` in the HARDA repo
5. Update `.env` → `YOLO_MODEL_PATH=ml/weights/harda_v1.pt`
6. The class-name mapping in `backend/app/services/yolo_detection.py` already
   covers RDD2022-style labels — no backend changes required.

## 1. Install dependencies

In [ ]:
!pip install --quiet ultralytics roboflow
import torch, ultralytics
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('ultralytics:', ultralytics.__version__)

## 2. Pull a labelled road-damage dataset from Roboflow

Replace `ROBOFLOW_API_KEY` with the value from your Roboflow account
(Settings → API → Private API Key). Two solid public starting points:

- **RDD2022** (Road Damage Dataset 2022, Maeda et al.) — most authoritative;
  classes: D00 longitudinal crack, D10 transverse crack, D20 alligator crack,
  D40 pothole. We collapse cracks → `uneven_surface` and D40 → `pothole`.
- **Roboflow Universe "pothole-and-road-defect"** — smaller, but already
  cleaned and includes lane-marking annotations in some versions.

Edit the workspace / project / version slugs below to match the dataset you
want to use. Keep `data.yaml` location reachable for the train step.

In [ ]:
ROBOFLOW_API_KEY = 'PASTE_YOUR_KEY_HERE'
WORKSPACE = 'roboflow-100'
PROJECT   = 'road-damage'           # change to the dataset you chose
VERSION   = 1                       # latest version number

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download('yolov8')
DATA_YAML = f'{dataset.location}/data.yaml'
print('Dataset downloaded to:', dataset.location)
print('Use this for training:', DATA_YAML)

## 3. Re-map class names to the HARDA taxonomy (optional)

If the source dataset uses raw labels like `D00`, `D10`, `D40`, the backend
already maps them via `_CLASS_MAP` in `yolo_detection.py`. You can either:
(a) leave the labels as-is and let the backend map them at inference time, or
(b) rewrite `data.yaml` to use HARDA-canonical names so the trained model
speaks the same vocabulary as the rest of the system.

(b) gives cleaner output but requires the source labels to be re-mapped on
disk — only do this if the dataset has 3 distinct underlying classes. For
RDD2022 specifically, (a) is the pragmatic choice.

In [ ]:
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print('Classes in dataset:', cfg.get('names'))
print('Total classes (nc):', cfg.get('nc'))

## 4. Fine-tune YOLOv8s

Starts from the pretrained COCO weights and fine-tunes for 50 epochs at
640×640 resolution. Batch size 16 is safe on a T4 (16 GB VRAM).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name='harda_v1',
    project='runs/detect',
    patience=10,           # early-stop if val mAP plateaus
    optimizer='AdamW',
    lr0=1e-3,
    seed=42,
    plots=True,            # writes confusion_matrix.png etc to runs/detect/harda_v1
)

## 5. Validate on the holdout set

Produces mAP@0.5, mAP@0.5:0.95, per-class precision/recall, and a confusion
matrix PNG. These numbers go straight onto the Progress 2 slide deck.

In [ ]:
metrics = model.val()
print(f"mAP@0.5      = {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 = {metrics.box.map:.4f}")
print('\nPer-class results:')
for i, name in enumerate(model.names.values()):
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0
    print(f"  {name:25s} precision={p:.3f}  recall={r:.3f}")

## 6. Quick sanity-check inference

Drop a sample road image into Colab (or use one from the val split) and
verify the trained model fires on it.

In [ ]:
import glob
samples = glob.glob(f'{dataset.location}/valid/images/*.jpg')[:3]
for s in samples:
    r = model.predict(s, conf=0.25, save=True)[0]
    print(s, '→', [r.names[int(c)] for c in r.boxes.cls])

## 7. Export `harda_v1.pt`

Download `runs/detect/harda_v1/weights/best.pt` from the Colab file panel,
rename to `harda_v1.pt`, drop it into `ml/weights/` in the repo, then update
`.env`:

```env
YOLO_MODEL_PATH=ml/weights/harda_v1.pt
```

Restart the Flask backend. `GET /api/v1/detection/model-info` should now
show the new path and the full class list.

In [ ]:
from google.colab import files
files.download('runs/detect/harda_v1/weights/best.pt')